# Day 15 — pandas: Series & DataFrames
### Python for Data Science · Module 1 · Topic 1.14

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

**Session length:** 2 hours
**Format:** 90 min concepts + live coding · 30 min practice

| # | What we cover | Time |
|---|---|---|
| 1 | Why pandas — NumPy plus labels | 10 min |
| 2 | Series — and a surprise about `nan` | 25 min |
| 3 | DataFrame — creating, columns, one bracket or two | 30 min |
| 4 | Inspecting data | 20 min |
| 5 | Mini build: from a CSV to a report | 5 min |
| 6 | **Practice notebook (separate file)** | 30 min |

> **Why this matters.** Every dataset from here to the capstone will arrive as a DataFrame.
> Time spent getting fluent now is repaid every single session for the next three and a
> half months.

In [4]:
import pandas as pd
import numpy as np
print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 2.2.3 | numpy 1.26.4


> **A note on versions.** pandas 3 shows text columns as dtype `str`; pandas 2 shows them as
> `object`. Depending on your Colab version you may see either. They mean the same thing.

---
# 1. Why pandas

## 1.1 What NumPy cannot do for you

In [2]:
# A NumPy array has no names
marks = np.array([[88, 71],
                  [91, 84],
                  [45, 38]])

print(marks[1, 0])        # which student? which subject? you have to remember

91


In [6]:
# A DataFrame names everything
df = pd.DataFrame({
    "name":   ["Ravi", "Sara", "Amit"],
    "python": [88, 91, 45],
    "stats":  [71, 84, 38],
    "Chemistry" : [72,91,64]
})
df

,name,python,stats,Chemistry
0,Ravi,88,71,72
1,Sara,91,84,91
2,Amit,45,38,64


> **The whole idea in one sentence.** A DataFrame is a collection of NumPy arrays — one per
> column — with names attached to the columns and labels attached to the rows.
>
> Everything from Days 12 to 14 still works underneath: broadcasting, masks, the axis rule.

---
# 2. Series

## 2.1 A Series is a NumPy array with labels

In [7]:
s = pd.Series([88, 71, 64])
print(s)
print()
print("values:", s.values, "  type:", type(s.values).__name__)   # a real NumPy array
print("index :", list(s.index))

0    88
1    71
2    64
dtype: int64

values: [88 71 64]   type: ndarray
index : [0, 1, 2]


In [9]:
# Your own labels
s = pd.Series([88, 71, 64], index=["Ravi", "Sara", "Amit"])
print(s)
print()
print('s["Sara"] =', s["Sara"])

# From a dict - the keys become the index
print()
print(pd.Series({"Ravi": 88, "Sara": 71}))

Ravi    88
Sara    71
Amit    64
dtype: int64

s["Sara"] = 71

Ravi    88
Sara    71
dtype: int64


In [10]:
# It behaves like Day 5's dictionary AND Day 12's array
s = pd.Series([88, 71, 64], index=["Ravi", "Sara", "Amit"])

print("look up by label :", s["Sara"])       # like a dict
print("vectorised       :\n", s * 2)         # like an array
print("boolean mask     :\n", s[s > 70])     # like an array

look up by label : 71
vectorised       :
 Ravi    176
Sara    142
Amit    128
dtype: int64
boolean mask     :
 Ravi    88
Sara    71
dtype: int64


## 2.2 ⚠️ The surprise — pandas skips `nan` by default

Yesterday: *"What does one `nan` do to `d.mean()`?"* → the whole result becomes `nan`.

**That is true in NumPy. It is not true in pandas.**

In [11]:
arr = np.array([88., 71., np.nan, 64.])

print("NumPy  mean :", arr.mean())
print("pandas mean :", pd.Series(arr).mean())
print("pandas mean, skipna=False :", pd.Series(arr).mean(skipna=False))

NumPy  mean : nan
pandas mean : 74.33333333333333
pandas mean, skipna=False : nan


In [12]:
# And the ddof default differs too
arr = np.array([88., 71., 64.])

print("NumPy  std :", arr.std().round(4), "  (ddof=0, population)")
print("pandas std :", pd.Series(arr).std().round(4), "  (ddof=1, sample)")

NumPy  std : 10.0775   (ddof=0, population)
pandas std : 12.3423   (ddof=1, sample)


| | NumPy | pandas |
|---|---|---|
| `nan` in a mean | returns `nan` | **skips it** |
| `std` default | `ddof=0` | `ddof=1` |
| to change it | `np.nanmean` | `skipna=False` |

**Why pandas made this choice:** it was built for real tables, where missing values are
normal. Returning `nan` for every column with one gap would make it almost useless.

> ### ⚠️ Convenient — and quietly dangerous
> NumPy's loud `nan` told you something was missing. pandas stays silent, so a mean over 3
> of 4 values looks exactly like a mean over all 4. **Yesterday's rule matters MORE in
> pandas, not less:** always check how many values a summary actually used.

In [13]:
s = pd.Series([88., 71., np.nan, 64.])

print("mean           :", s.mean().round(2))
print("computed from  :", s.count(), "of", len(s), "values")   # count() skips NaN
print("missing        :", s.isna().sum())

mean           : 74.33
computed from  : 3 of 4 values
missing        : 1


---
# 3. DataFrame

## 3.1 Three ways to build one

In [39]:
# 1. A dict of lists - one list per COLUMN (the most common)
df = pd.DataFrame({
    "name":   ["Ravi", "Sara", "Amit", "Neha"],
    "city":   ["Pune", "Mumbai", "Delhi", "Pune"],
    "python": [88, 91, 45, 67],
    "stats":  [71, 84, 38, 73],
})
df

,name,city,python,stats
0,Ravi,Pune,88,71
1,Sara,Mumbai,91,84
2,Amit,Delhi,45,38
3,Neha,Pune,67,73


In [15]:
# 2. A list of dicts - one dict per ROW
rows = [
    {"name": "Ravi", "python": 88},
    {"name": "Sara", "python": 91},
]
pd.DataFrame(rows)

# Natural when rows arrive one at a time - Day 9's DictReader gave you exactly this.

,name,python
0,Ravi,88
1,Sara,91


In [16]:
# 3. From a file - the one you will use most.
#    First make a small CSV, including a quoted comma and a blank cell.
with open("marks.csv", "w") as f:
    f.write("name,city,mark\n")
    f.write("Ravi,Pune,88\n")
    f.write('Sara,"Mumbai, MH",91\n')
    f.write("Amit,Delhi,\n")                 # blank mark

d = pd.read_csv("marks.csv")
d

,name,city,mark
0,Ravi,Pune,88.0
1,Sara,"Mumbai, MH",91.0
2,Amit,Delhi,NaN


**Remember the Day 9 promise** — one line to replace everything you did by hand:

- the quoted comma in `"Mumbai, MH"` was handled correctly
- the blank mark became `NaN`
- the `mark` column became `float64`, because an int column cannot hold `NaN` (Day 14)

In [17]:
print(d.dtypes)
print()
print("missing per column:")
print(d.isna().sum())

name     object
city     object
mark    float64
dtype: object

missing per column:
name    0
city    0
mark    1
dtype: int64


## 3.2 The anatomy of a DataFrame

In [18]:
print("shape  :", df.shape)
print("columns:", list(df.columns))
print("index  :", df.index)
print()
print(df.dtypes)

shape  : (4, 4)
columns: ['name', 'city', 'python', 'stats']
index  : RangeIndex(start=0, stop=4, step=1)

name      object
city      object
python     int64
stats      int64
dtype: object


**Each column has its own type.** That is the difference from a NumPy array, which forces
one type on everything.

## 3.3 One bracket or two — and what comes back

In [19]:
# One column -> a Series
col = df["python"]
print(type(col).__name__)
print(col)

Series
0    88
1    91
2    45
3    67
Name: python, dtype: int64


In [35]:
# Several columns -> a DataFrame  (DOUBLE brackets)
sub = df[["name", "python","stats"]]
print(type(sub).__name__)
sub

DataFrame


,name,python,stats
0,Ravi,88,71
1,Sara,91,84
2,Amit,45,38
3,Neha,67,73


In [27]:
# Even ONE name in double brackets gives a DataFrame
print(type(df["python"]).__name__)      # Series
print(type(df[["python"]]).__name__)    # DataFrame

# The outer brackets select. The inner brackets are a Python list of names.

Series
DataFrame


### ⚠️ Two ways this goes wrong

In [ ]:
# 1. Forgetting the inner brackets
try:
    df["name", "python"]
except KeyError:
    print('df["name", "python"] -> KeyError')
    print("   Python reads that as ONE key: the tuple ('name', 'python')")

In [37]:
# 2. Attribute access works - until it does not
d2 = pd.DataFrame({"count": [1, 2], "score": [3, 4]})

print("d2.score      :", d2.score.tolist())         # fine
print("type(d2.count):", type(d2.count).__name__)   # a METHOD, not your column!
print('d2["count"]   :', d2["count"].tolist())      # always works

# Prefer df["col"]. Names like count, sum, mean and max are all DataFrame methods,
# and names with spaces cannot be written as attributes at all.

d2.score      : [3, 4]
type(d2.count): method
d2["count"]   : [1, 2]


## 3.4 Adding columns — vectorised, exactly like NumPy

In [47]:
df["Chemistry"] = [45,78,82,65]
df["total"]   = df["python"] + df["stats"] + df["Chemistry"]   # Day 13's element-wise arithmetic
df["average"] = df["total"] / 3              # broadcasting a scalar
df["passed"]  = df["total"] >= 180
df["remarks"] = np.where(df["total"] >= 200, "Best Score",
              np.where(df["total"] >= 170, "Improvement Needed", "Meet me"))           # a boolean column, like a Day 12 mask
df

,name,city,python,stats,Chemistry,total,average,passed,grade,remarks
0,Ravi,Pune,88,71,45,204,68.000000,True,Improvement Needed,Best Score
1,Sara,Mumbai,91,84,78,253,84.333333,True,Best Score,Best Score
2,Amit,Delhi,45,38,82,165,55.000000,False,Meet me,Meet me
3,Neha,Pune,67,73,65,205,68.333333,True,Improvement Needed,Best Score


In [45]:
# np.where from Day 13 works on columns too
df["grade"] = np.where(df["average"] >= 80, "Best Score",
              np.where(df["average"] >= 60, "Improvement Needed", "Meet me"))
df[["name", "average", "grade"]]

,name,average,grade
0,Ravi,68.000000,Improvement Needed
1,Sara,84.333333,Best Score
2,Amit,55.000000,Meet me
3,Neha,68.333333,Improvement Needed


### ⚠️ Never loop over the rows to build a column

In [ ]:
import time

big = pd.DataFrame({"a": np.random.rand(100_000), "b": np.random.rand(100_000)})

# The loop way
t = time.time()
out = []
for i in range(len(big)):
    out.append(big["a"].iloc[i] + big["b"].iloc[i])
loop_time = time.time() - t

# The whole-column way
t = time.time()
out = big["a"] + big["b"]
vec_time = time.time() - t

print(f"loop    : {loop_time*1000:8.1f} ms")
print(f"column  : {vec_time*1000:8.1f} ms")
print(f"about {loop_time/vec_time:.0f}x faster")

> **What about `.apply()`?** You will see `df["col"].apply(some_function)` everywhere online.
> It runs a Python function once per value — flexible, and fine when no vectorised version
> exists, but it is still a loop underneath. Reach for whole-column arithmetic and
> `np.where` first; use `.apply` when they genuinely cannot express what you need.

---
# 4. Inspecting data — the first five commands

| Command | Tells you |
|---|---|
| `df.head()` | the first 5 rows |
| `df.shape` | `(rows, columns)` |
| `df.info()` | types and **non-null counts** |
| `df.describe()` | count, mean, std, quartiles — per column |
| `df["col"].value_counts()` | how often each value appears |

In [ ]:
df.head()

,name,city,python,stats,Chemistry,total,average,passed,grade,remarks
0,Ravi,Pune,88,71,45,204,68.000000,True,Improvement Needed,Best Score
1,Sara,Mumbai,91,84,78,253,84.333333,True,Best Score,Best Score
2,Amit,Delhi,45,38,82,165,55.000000,False,Meet me,Meet me
3,Neha,Pune,67,73,65,205,68.333333,True,Improvement Needed,Best Score


In [ ]:
df.tail()

,name,city,python,stats,Chemistry,total,average,passed,grade,remarks
1,Sara,Mumbai,91,84,78,253,84.333333,True,Best Score,Best Score
2,Amit,Delhi,45,38,82,165,55.000000,False,Meet me,Meet me
3,Neha,Pune,67,73,65,205,68.333333,True,Improvement Needed,Best Score


In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   name       4 non-null      object 
 1   city       4 non-null      object 
 2   python     4 non-null      int64  
 3   stats      4 non-null      int64  
 4   Chemistry  4 non-null      int64  
 5   total      4 non-null      int64  
 6   average    4 non-null      float64
 7   passed     4 non-null      bool   
 8   grade      4 non-null      object 
 9   remarks    4 non-null      object 
dtypes: bool(1), float64(1), int64(4), object(4)
memory usage: 420.0+ bytes


In [60]:
df.describe().round(2)

# describe's count row is the number of NON-missing values.
# Its std uses ddof=1 - pandas' default, not NumPy's.

,python,stats,Chemistry,total,average
count,4.00,4.00,4.00,4.00,4.00
mean,72.75,66.50,67.50,206.75,68.92
std,21.36,19.84,16.66,36.02,12.01
min,45.00,38.00,45.00,165.00,55.00
25%,61.50,62.75,60.00,194.25,64.75
50%,77.50,72.00,71.50,204.50,68.17
75%,88.75,75.75,79.00,217.00,72.33
max,91.00,84.00,82.00,253.00,84.33


In [ ]:
df["city"].value_counts()

# Day 5's counting pattern, in one method. Great for spotting typos
# in a text column: "Pune" and "pune" would appear as separate rows.

---
# 5. Putting it together — from a CSV to a report

In [ ]:
# Build the input file
pd.DataFrame({
    "name":   ["Ravi", "Sara", "Amit", "Neha"],
    "city":   ["Pune", "Mumbai", "Delhi", "Pune"],
    "python": [88, 91, 45, 67],
    "stats":  [71, 84, 38, 73],
}).to_csv("class.csv", index=False)

df = pd.read_csv("class.csv")               # one line, Day 9's promise

# ---- 1. inspect before trusting anything
print(df.shape)
print(df.isna().sum().to_dict())

# ---- 2. derive, without a loop
df["total"]   = df["python"] + df["stats"]
df["average"] = df["total"] / 2
df["result"]  = np.where(df["average"] >= 50, "pass", "fail")

# ---- 3. summarise, honestly
print(df[["python", "stats", "average"]].describe().round(1))
print(df["result"].value_counts())

# ---- 4. save a NEW file - never overwrite the input
df.to_csv("class_report.csv", index=False)

In [ ]:
# Why index=False matters
df.to_csv("with_index.csv")                  # forgot index=False
print(pd.read_csv("with_index.csv").columns.tolist())

# The 0, 1, 2 index was written as an extra column, and it comes back
# as "Unnamed: 0". Always pass index=False unless the index means something.

Every tool in that build is one you already know:

- **`read_csv`** — Day 9's whole session, in one line
- **Whole-column arithmetic** — Day 13, with names on
- **`np.where`** — Day 13's vectorised if/else
- **`describe`** — Day 14's summary, every column at once
- **`value_counts`** — Day 5's counting pattern
- **`index=False`** — Day 9's rule: write a new file, cleanly

---
# 6. Recap — the twelve things to remember

1. `import pandas as pd` — always that alias.
2. A Series is a NumPy array with labels on it.
3. A DataFrame is NumPy columns with names on them.
4. Each column has its own type; an array has only one.
5. pandas **skips** `nan` by default. NumPy does not.
6. pandas `std` uses `ddof=1`; NumPy uses `ddof=0`.
7. `df["col"]` gives a Series; `df[["a", "b"]]` gives a DataFrame.
8. Prefer `df["col"]` over `df.col` — `count` and `sum` are methods.
9. New columns are whole-column expressions. Never loop.
10. Inspect first: `head`, `shape`, `info`, `describe`.
11. `info()` shows the non-null count — missing data at a glance.
12. `to_csv(..., index=False)` to avoid a stray index column.

---

### 📝 Now open **`Day15_Practice_Questions.ipynb`** for the 30-minute practice session.

### Homework
- Rebuild Day 9's CSV report in pandas and compare the line counts.
- Create a DataFrame of 5 things you own, with 4 columns of different types.
- Run the five inspection commands on any CSV you can find.

### Next class — Topic 1.15: Filtering, selection & indexing
`loc` versus `iloc`, boolean filtering on a DataFrame, and setting a meaningful index.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*